# Etapa 2 – Pré-Processamento de Dados
Este notebook realiza o pré-processamento completo do dataset **students_performance.csv**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import joblib

df = pd.read_csv("students_performance.csv")
df.head()

,id_estudante,idade,genero,escolaridade_pais,horas_estudo_semanal,frequencia_escolar,extracurricular,horas_de_sono,notas_anteriores,aula_reforco,qualidade_internet,renda_familiar,estado_de_saude,nota_final
0,STD01416,22,M,bachelor,5.66,69.55,Yes,6.49,49.41,No,Good,Medium,Good,85.39
1,STD01345,19,M,bachelor,13.30,58.82,Yes,8.12,50.29,No,NaN,Medium,Good,98.43
2,STD01705,25,M,master,10.43,59.72,Yes,6.60,71.64,No,Good,Medium,Good,100.00
3,STD00299,21,F,bachelor,3.90,68.33,Yes,6.99,50.93,Yes,Poor,Low,Excellent,86.85
4,STD01762,19,F,bachelor,4.24,50.44,Yes,8.06,54.00,No,Poor,Medium,Excellent,83.25


## 1. Tratamento de Valores Faltantes
Nesta seção tratamos valores ausentes em colunas numéricas e categóricas.

In [2]:
# Separando colunas numéricas e categóricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Imputação numérica (mediana)
imputer_num = SimpleImputer(strategy="median")
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# Imputação categórica (moda)
imputer_cat = SimpleImputer(strategy="most_frequent")
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

df.head()

,id_estudante,idade,genero,escolaridade_pais,horas_estudo_semanal,frequencia_escolar,extracurricular,horas_de_sono,notas_anteriores,aula_reforco,qualidade_internet,renda_familiar,estado_de_saude,nota_final
0,STD01416,22.0,M,bachelor,5.66,69.55,Yes,6.49,49.41,No,Good,Medium,Good,85.39
1,STD01345,19.0,M,bachelor,13.30,58.82,Yes,8.12,50.29,No,Good,Medium,Good,98.43
2,STD01705,25.0,M,master,10.43,59.72,Yes,6.60,71.64,No,Good,Medium,Good,100.00
3,STD00299,21.0,F,bachelor,3.90,68.33,Yes,6.99,50.93,Yes,Poor,Low,Excellent,86.85
4,STD01762,19.0,F,bachelor,4.24,50.44,Yes,8.06,54.00,No,Poor,Medium,Excellent,83.25


In [8]:
df.isnull().sum()

idade                   0
horas_estudo_semanal    0
frequencia_escolar      0
horas_de_sono           0
notas_anteriores        0
                       ..
estado_de_saude_GOOD    0
estado_de_saude_Good    0
estado_de_saude_POOR    0
estado_de_saude_Poor    0
performance_ratio       0
Length: 2567, dtype: int64

## 2. Tratamento de Outliers
Aplicamos capping (limite) usando IQR para reduzir impacto de valores extremos.

In [9]:
df.shape

(2510, 2567)

In [3]:
# Função para capping via IQR
def cap_outliers(col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower, lower,
                       np.where(df[col] > upper, upper, df[col]))

for col in num_cols:
    cap_outliers(col)

df.describe()

,idade,horas_estudo_semanal,frequencia_escolar,horas_de_sono,notas_anteriores,nota_final
count,2510.000000,2510.000000,2510.00000,2510.000000,2510.000000,2510.000000
mean,21.596414,10.067461,59.63483,6.986008,57.299335,92.096131
std,2.301174,4.521812,9.62838,1.156683,9.078007,7.425408
min,17.000000,-1.896250,35.31750,4.100000,33.908750,67.681250
25%,20.000000,6.980000,53.71500,6.260000,51.522500,86.982500
50%,22.000000,9.960000,59.90500,6.990000,57.300000,93.310000
75%,24.000000,12.897500,65.98000,7.700000,63.265000,99.850000
max,30.000000,21.773750,84.37750,9.860000,80.878750,101.070000


## 3. Encoding de Variáveis Categóricas
Convertendo texto para números usando One-Hot Encoding.

In [4]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df.head()

,idade,horas_estudo_semanal,frequencia_escolar,horas_de_sono,notas_anteriores,nota_final,id_estudante_STD00002,id_estudante_STD00003,id_estudante_STD00004,id_estudante_STD00005,...,renda_familiar_MEDIUM,renda_familiar_Medium,estado_de_saude_ Good,estado_de_saude_ Poor,estado_de_saude_EXCELLENT,estado_de_saude_Excellent,estado_de_saude_GOOD,estado_de_saude_Good,estado_de_saude_POOR,estado_de_saude_Poor
0,22.0,5.66,69.55,6.49,49.41,85.39,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
1,19.0,13.30,58.82,8.12,50.29,98.43,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
2,25.0,10.43,59.72,6.60,71.64,100.00,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
3,21.0,3.90,68.33,6.99,50.93,86.85,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
4,19.0,4.24,50.44,8.06,54.00,83.25,False,False,False,False,...,False,True,False,False,False,True,False,False,False,False


## 4. Normalização das Variáveis Numéricas
Aplicamos StandardScaler e salvamos o scaler.

In [5]:
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

joblib.dump(scaler, "scaler.pkl")

df.head()

,idade,horas_estudo_semanal,frequencia_escolar,horas_de_sono,notas_anteriores,nota_final,id_estudante_STD00002,id_estudante_STD00003,id_estudante_STD00004,id_estudante_STD00005,...,renda_familiar_MEDIUM,renda_familiar_Medium,estado_de_saude_ Good,estado_de_saude_ Poor,estado_de_saude_EXCELLENT,estado_de_saude_Excellent,estado_de_saude_GOOD,estado_de_saude_Good,estado_de_saude_POOR,estado_de_saude_Poor
0,0.175417,-0.974905,1.029991,-0.428905,-0.869233,-0.903313,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
1,-1.128525,0.715019,-0.084645,0.980578,-0.772277,0.853169,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
2,1.479359,0.080192,0.008848,-0.333786,1.580030,1.064648,False,False,False,False,...,False,True,False,False,False,False,False,True,False,False
3,-0.259230,-1.364208,0.903257,0.003452,-0.701762,-0.706652,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
4,-1.128525,-1.289001,-0.955162,0.928696,-0.363515,-1.191570,False,False,False,False,...,False,True,False,False,False,True,False,False,False,False


## 5. Feature Engineering (Opcional)
Criamos novas features para enriquecer o dataset.

In [6]:
# Exemplo: criar uma feature 'performance_ratio'
# Evite divisões por zero
df["performance_ratio"] = df["nota_final"] / (df["horas_estudo_semanal"] + 1)

df.head()

,idade,horas_estudo_semanal,frequencia_escolar,horas_de_sono,notas_anteriores,nota_final,id_estudante_STD00002,id_estudante_STD00003,id_estudante_STD00004,id_estudante_STD00005,...,renda_familiar_Medium,estado_de_saude_ Good,estado_de_saude_ Poor,estado_de_saude_EXCELLENT,estado_de_saude_Excellent,estado_de_saude_GOOD,estado_de_saude_Good,estado_de_saude_POOR,estado_de_saude_Poor,performance_ratio
0,0.175417,-0.974905,1.029991,-0.428905,-0.869233,-0.903313,False,False,False,False,...,True,False,False,False,False,False,True,False,False,-35.996420
1,-1.128525,0.715019,-0.084645,0.980578,-0.772277,0.853169,False,False,False,False,...,True,False,False,False,False,False,True,False,False,0.497469
2,1.479359,0.080192,0.008848,-0.333786,1.580030,1.064648,False,False,False,False,...,True,False,False,False,False,False,True,False,False,0.985610
3,-0.259230,-1.364208,0.903257,0.003452,-0.701762,-0.706652,False,False,False,False,...,False,False,False,False,True,False,False,False,False,1.940244
4,-1.128525,-1.289001,-0.955162,0.928696,-0.363515,-1.191570,False,False,False,False,...,True,False,False,False,True,False,False,False,False,4.123059


In [7]:
import os

# Garantir que a pasta data existe
os.makedirs("../data", exist_ok=True)

# Salvar o dataset preprocessado
df.to_csv("../data/dados_preprocessados.csv", index=False)

print("Arquivo salvo em ../data/dados_preprocessados.csv")

Arquivo salvo em ../data/dados_preprocessados.csv
